# A/B Testing

Below, we identify the control and variant groups, calculate the key metrics (conversion rate, ARPU, ARPPU) and perform power analysis

In [1]:
# Create paths to pkl files
import os
project_root = os.path.abspath("..")
ab_path = os.path.join(project_root, "data", "processed", "clean_ab_test.pkl")

In [2]:
# Import library
import pandas as pd
import numpy as np
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

In [3]:
df_ab_test = pd.read_pickle(ab_path)
df_ab_test.head()

,user_id,revenue,testgroup
0,1,0,b
1,2,0,a
2,3,0,a
3,4,0,b
4,5,0,b


## Descriptive analysis

Let's get some intuition prior to formal testing! 

We calculate basic metrics (conversion rate, ARPU, ARPPU) for each group to understand possible differences and potential patterns. We also attempt to provide possible, naive explanations to the observed phenomena.

In [4]:
metrics = df_ab_test.groupby(by="testgroup").agg(
    no_of_players = ("user_id", "nunique"),
    no_of_payers = ("revenue", lambda x: (x > 0).sum()),
    conversion_rate = ("revenue", lambda x: (x > 0).sum() / x.count() *100),
    total_revenue = ("revenue", "sum"),
    arpu = ("revenue", lambda x: x.mean()),
    arppu = ("revenue", lambda x: x[(x > 0)].mean())
).reset_index("testgroup")
metrics

,testgroup,no_of_players,no_of_payers,conversion_rate,total_revenue,arpu,arppu
0,a,202103,1928,0.953969,5136189,25.413720,2663.998444
1,b,202667,1805,0.890624,5421603,26.751287,3003.658172


Observations: 
- It is said that the test group has higher arpu => group b is the test/variant group.
- Promotion A has higher conversion rate => It is better at converting players into paying players.
- Promotion B has higher arpu => It is better at attracting heavy spenders.
- Promotion B also has higher arppu => It should be better at encouraging multiple purchases.

So if we want to grow the base of consistently paying players (like a subscription-based system), promotion A should be chosen. If we want higher revenue, we should go with promotion B.

Having said that, there are other things we should consider: 
- Promotion B may bring higher revenue because it is priced at a higher price than promotion A, which may be a good explanation for the lower rate of paying players yet higher revenue in B.
- Even though promotion A results in less revenue overall, it is still useful in getting players into being comfortable with purchasing. From there, a different strategy can be utilized to increase revenue. 

## Power Analysis - Calculating required sample size

Let's determine whether the A/B test is adequately powered!

We estimate the minimum required sample size using power analysis tools, then compare it to the actual sample sizes.

In [5]:
import statsmodels.stats.api as sms
effect_size =  sms.proportion_effectsize(0.89, 0.95)

What is the power to detect an effect of size d = -0.23 with sample size n1 = 202103 and n2 = 202667, and at the alpha = 0.05 level?

In [6]:
# 2 sample sizes
from statsmodels.stats.power import TTestIndPower
analysis = TTestIndPower()

# single sample size
# from statsmodels.stats.power import TTestPower
# analysis = TTestPower()

# solve for power
power = analysis.solve_power(effect_size=effect_size, alpha=0.05, nobs1=202103, ratio=202667/202103, power=None, alternative='two-sided')
print('power = ' + str(power*100) + '%')

power = 100.0%


The A/B test has enough participants for us to be 100% sure of its outcome at confidence level of 0.05

In [7]:
import statsmodels.stats.api as sms

sensitivity = 1
confidence_level = 0.05
sample_size = sms.NormalIndPower().solve_power(effect_size=effect_size, power=sensitivity, alpha=confidence_level, ratio=202667/202103)
print('Required sample size is', round(sample_size))

Required sample size is 5000


Indeed, we only need 5000 participants to be 100% sure of rejeting the null hypothesis.

## Hypothesis Testing

### Conversion rate

Let's see whether different promotions affect the probability of converting players into paying players!

Conversion is binary, so we can use the standard t-test for it!

In [8]:
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import proportion_confint

control_total = int(metrics.loc[metrics["testgroup"] == "a", "no_of_players"])
variant_total = int(metrics.loc[metrics["testgroup"] == "b", "no_of_players"])
nobs = [control_total, variant_total]

control_converted = int(metrics.loc[metrics["testgroup"] == "a", "no_of_payers"])
variant_converted = int(metrics.loc[metrics["testgroup"] == "b", "no_of_payers"])
converted = [control_converted, variant_converted]

# p-value
stat, pval = proportions_ztest(converted, nobs=nobs)
print('p-value: ', round(pval,3))

# confidence interval
(lower_con, lower_var), (upper_con, upper_var) = proportion_confint(
    converted, nobs = nobs, alpha=0.05
)
print(f'95% confidence interval for variant group: [{lower_var:.3f}, {upper_var:.3f}]')

p-value:  0.035
95% confidence interval for variant group: [0.008, 0.009]


With p-value = 0.035 < 0.05, promotion A/the control group has statistically significantly higher conversion rate. 

The entire confidence interval is above 0, the uplift is statistically significant and precisely estimated to be between 0.8% and 0.9%.

> This is the result of our very large test size (over 200000 for each test group)

### ARPU

ARPU is continuous and skewed, so we cannot use the standard t-test. Instead, let's try implementing bootstrap and two-part model!

#### ARPU - Bootstrap inference

In [9]:
# get the revenue of each group
revenue_a = df_ab_test[df_ab_test["testgroup"] == 'a']["revenue"].values
revenue_b = df_ab_test[df_ab_test["testgroup"] == 'b']["revenue"].values

In [22]:
seed = 13
rng = np.random.default_rng(seed)

n_boots = 5000 # number of simulations
boot_diffs = np.zeros(n_boots) # zero array containing diff values

for i in range(n_boots):
    sample_a = rng.choice(revenue_a, size=len(revenue_a), replace=True) # sample with replacement for group a
    sample_b = rng.choice(revenue_b, size=len(revenue_b), replace=True) # sample with replacement for group b
    boot_diffs[i] = sample_b.mean() - sample_a.mean() # difference between the means of 2 groups after each simulation

# compute 95% confidence interval
lower_ci, upper_ci = np.percentile(boot_diffs, [2.5, 97.5])
print(f'95% confidence interval for variant group: [{lower_ci:.3f}, {upper_ci:.3f}]')

# compute p-value
p_val = 2 * min(
    (boot_diffs <= 0).mean(),
    (boot_diffs >= 0).mean()
)
print('p-value: ', round(p_val,3))

# compute uplife
arpu_a = revenue_a.mean() 
arpu_b = revenue_b.mean() 
uplift = arpu_b - arpu_a
print('uplift: ', round(uplift, 2))

95% confidence interval for variant group: [-2.930, 5.423]
p-value:  0.542
uplift:  1.34


Observations:
- Promotion B may increase ARPU by 1.34, but with the 95% confidence interval including 0, the true effect can be lower (-2.93) or higher (5.423) ARPU => we don't have enough evidence to confirm with 95% confidence that promotion B affects ARPU.

#### ARPU - Two-part model / P[Y > 0] * E[Y | Y > 0]

In [11]:
# create binary variables for variant group and converted users, and log the revenue
df_ab_test["variant"] = (df_ab_test["testgroup"] == "b").astype(int)
df_ab_test["converted"] = (df_ab_test["revenue"] > 0).astype(int)
df_ab_test["log_revenue"] = np.log(df_ab_test["revenue"].where(df_ab_test["revenue"] > 0, np.nan))
df_ab_test

,user_id,revenue,testgroup,variant,converted,log_revenue
0,1,0,b,1,0,NaN
1,2,0,a,0,0,NaN
2,3,0,a,0,0,NaN
3,4,0,b,1,0,NaN
4,5,0,b,1,0,NaN
...,...,...,...,...,...,...
404765,404766,0,a,0,0,NaN
404766,404767,0,b,1,0,NaN
404767,404768,231,a,0,1,5.442418
404768,404769,0,a,0,0,NaN


In [12]:
# Logistic regression - Payer probablity P[Y > 0]
import statsmodels.formula.api as smf

logit_model = smf.logit("converted ~ variant", data=df_ab_test).fit()
print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.052392
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:              converted   No. Observations:               404770
Model:                          Logit   Df Residuals:                   404768
Method:                           MLE   Df Model:                            1
Date:                Sun, 19 Apr 2026   Pseudo R-squ.:               0.0001048
Time:                        20:51:36   Log-Likelihood:                -21207.
converged:                       True   LL-Null:                       -21209.
Covariance Type:            nonrobust   LLR p-value:                   0.03501
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -4.6427      0.023   -202.882      0.000      -4.688      -4.598
variant       -0.0693      0.

The coefficient of variant = -0.07 and p-value = 0.035
> Promotion B has statistically lower chance to convert players into payers.

In [13]:
# Gamme regression - Payer spend E[Y | Y > 0]
import statsmodels.api as sm
gamma_model = smf.glm(
    formula="revenue ~ variant",
    data=df_ab_test[df_ab_test["converted"] == 1],
    family=sm.families.Gamma(sm.families.links.log())
).fit()

print(gamma_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                revenue   No. Observations:                 3733
Model:                            GLM   Df Residuals:                     3731
Model Family:                   Gamma   Df Model:                            1
Link Function:                    log   Scale:                          5.9768
Method:                          IRLS   Log-Likelihood:                -34739.
Date:                Sun, 19 Apr 2026   Deviance:                       7342.1
Time:                        20:51:37   Pearson chi2:                 2.23e+04
No. Iterations:                     9   Pseudo R-squ. (CS):          0.0006019
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      7.8876      0.056    141.665      0.0

The coefficient of variant = 0.12 and p-value = 0.134
> There is chance that payers in promotion B spend more, but it is not statistically significant.

In [17]:
# get the revenue of each group
revenue_payer_a = df_ab_test[(df_ab_test["testgroup"] == 'a') & (df_ab_test["revenue"] > 0)]["revenue"].values
revenue_payer_b = df_ab_test[(df_ab_test["testgroup"] == 'b') & (df_ab_test["revenue"] > 0)]["revenue"].values

In [ ]:
seed = 13
rng = np.random.default_rng(seed)

n_boots = 5000 # number of simulations
boot_diffs = np.zeros(n_boots) # zero array containing diff values

for i in range(n_boots):
    sample_a = rng.choice(revenue_payer_a, size=len(revenue_payer_a), replace=True) # sample with replacement for group a
    sample_b = rng.choice(revenue_payer_b, size=len(revenue_payer_b), replace=True) # sample with replacement for group b
    boot_diffs[i] = sample_b.mean() - sample_a.mean() # difference between the means of 2 groups after each simulation

# compute 95% confidence interval
lower_ci, upper_ci = np.percentile(boot_diffs, [2.5, 97.5])
print(f'95% confidence interval for variant group: [{lower_ci:.3f}, {upper_ci:.3f}]')

# compute p-value
p_val = 2 * min(
    (boot_diffs <= 0).mean(),
    (boot_diffs >= 0).mean()
)
print('p-value: ', round(p_val,3))

# compute uplife
arppu_a = revenue_payer_a.mean() 
arppu_b = revenue_payer_b.mean() 
uplift = arppu_b - arppu_a
print('uplift: ', round(uplift, 2))

95% confidence interval for variant group: [-65.840, 727.503]
p-value:  0.098
uplift:  339.66


Observations:
- Promotion B may increase ARPPU by 339.66, but with the 95% confidence interval including 0, the true effect can be lower (-65.84) or higher (+727.503) ARPPU => we don't have enough evidence to confirm with 95% confidence that promotion B affects ARPU.